# 🏥 Clinisight AI — PubMed Evidence Fetcher Demo

This notebook demonstrates how the **PubMed article fetcher** works step by step:

1. **Query Building** — How raw symptoms become optimized PubMed search queries
2. **Article Search** — Hitting the PubMed E-utilities API to find article IDs
3. **Article Fetching** — Retrieving full metadata (title, abstract, authors, date)
4. **Text Formatting** — Converting article dicts into clean LLM-ready text
5. **Full Pipeline** — End-to-end: symptoms → articles → summary

---
## Step 0: Setup & Imports

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from functions.pubmed_articles import (
    _build_pubmed_query,
    fetch_pubmed_articles_with_metadata,
    format_articles_as_text,
)

print("✅ All imports successful!")

---
## Step 1: Query Building

Raw symptom text like `"fever headache nausea"` is **not a good PubMed query**.  
Our query builder converts it into a proper boolean search with clinical context filters.

In [ ]:
# See how raw symptom text gets transformed into optimized PubMed queries

test_queries = [
    "fever headache",
    "severe chest pain, difficulty breathing",
    "nausea vomiting abdominal pain",
    ""  # empty input
]

for raw in test_queries:
    built = _build_pubmed_query(raw)
    print(f"Raw:   '{raw}'")
    print(f"Built: '{built}'")
    print()

---
## Step 2: Search PubMed for Article IDs

The first API call goes to PubMed's `esearch` endpoint. It returns **PubMed IDs (PMIDs)** — unique identifiers for each article.

In [ ]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

# We'll manually do what our function does internally
query = _build_pubmed_query("fever headache")
print(f"Search query: {query}\n")

search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
search_params = {
    "db": "pubmed",
    "term": query,
    "retmax": 3,
    "retmode": "json",
    "sort": "relevance",
}

# Add API key if available
ncbi_key = os.getenv("NCBI_API_KEY", "")
if ncbi_key:
    search_params["api_key"] = ncbi_key
    print("🔑 Using NCBI API key (10 req/sec rate limit)")
else:
    print("⚠️ No NCBI API key (3 req/sec rate limit)")

response = requests.get(search_url, params=search_params, timeout=15)
data = response.json()

id_list = data["esearchresult"]["idlist"]
total_found = data["esearchresult"]["count"]

print(f"\n📊 Total articles matching: {total_found}")
print(f"📋 Top {len(id_list)} PMIDs returned: {id_list}")

---
## Step 3: Fetch Full Article Metadata (XML)

Now we take those PMIDs and hit the `efetch` endpoint to get the **full XML** with title, abstract, authors, date, etc.

In [ ]:
from bs4 import BeautifulSoup
import warnings
from bs4 import XMLParsedAsHTMLWarning
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
fetch_params = {
    "db": "pubmed",
    "id": ",".join(id_list),
    "retmode": "xml",
}
if ncbi_key:
    fetch_params["api_key"] = ncbi_key

fetch_response = requests.get(fetch_url, params=fetch_params, timeout=15)
print(f"Response status: {fetch_response.status_code}")
print(f"XML length: {len(fetch_response.text)} characters\n")

# Show a snippet of raw XML
print("--- Raw XML snippet (first 500 chars) ---")
print(fetch_response.text[:500])
print("...")

---
## Step 4: Parse XML into Structured Data

We parse the raw XML using **BeautifulSoup** to extract clean structured data for each article.

In [ ]:
soup = BeautifulSoup(fetch_response.text, "lxml")
articles_xml = soup.find_all("pubmedarticle")

print(f"Found {len(articles_xml)} articles in XML\n")

# Parse each article manually to show the structure
for i, article in enumerate(articles_xml, 1):
    pmid = article.find("pmid").get_text(strip=True)
    title = article.find("articletitle").get_text(strip=True) if article.find("articletitle") else "N/A"
    abstract_tag = article.find("abstract")
    abstract = abstract_tag.get_text(separator=" ", strip=True)[:200] + "..." if abstract_tag else "No abstract"
    
    # Authors
    authors = []
    for author in article.find_all("author"):
        last = author.find("lastname")
        fore = author.find("forename")
        if last and fore:
            authors.append(f"{fore.get_text()} {last.get_text()}")
    
    # Date
    date_tag = article.find("pubdate")
    year = date_tag.find("year").get_text() if date_tag and date_tag.find("year") else "N/A"
    
    print(f"📄 Article {i}")
    print(f"   PMID:     {pmid}")
    print(f"   Title:    {title}")
    print(f"   Authors:  {', '.join(authors[:3])}{'...' if len(authors) > 3 else ''}")
    print(f"   Year:     {year}")
    print(f"   URL:      https://pubmed.ncbi.nlm.nih.gov/{pmid}/")
    print(f"   Abstract: {abstract}")
    print()

---
## Step 5: Using the Production Function

Now let's use the **actual hardened function** that does all of the above with retry logic, input validation & clean formatting.

In [ ]:
# Use the production function
articles = fetch_pubmed_articles_with_metadata("fever headache")

print(f"✅ Fetched {len(articles)} articles\n")

for i, art in enumerate(articles, 1):
    print(f"📄 Article {i}:")
    print(f"   Title:    {art['title']}")
    print(f"   Authors:  {', '.join(art['authors'][:3])}")
    print(f"   Date:     {art['publication_date']}")
    print(f"   URL:      {art['article_url']}")
    print(f"   Abstract: {art['abstract'][:150]}...")
    print()

---
## Step 6: Format for LLM Summarization

The `format_articles_as_text()` converts `list[dict]` → **clean text block** that the LLM can summarize properly (not raw Python dicts!).

In [ ]:
# Convert article dicts → clean formatted text
formatted_text = format_articles_as_text(articles)

print("--- Formatted Text (this is what gets sent to LLM for summarization) ---\n")
print(formatted_text)

---
## Step 7: Edge Cases — Empty & Invalid Queries

In [ ]:
# Test: Empty query
print("=" * 50)
print("Test: Empty query")
print("=" * 50)
result = fetch_pubmed_articles_with_metadata("")
print(f"Result: {result}")
print(f"Formatted: {format_articles_as_text(result)}")
print()

# Test: Whitespace-only input
print("=" * 50)
print("Test: Whitespace-only query")
print("=" * 50)
result = fetch_pubmed_articles_with_metadata("   ")
print(f"Result: {result}")
print(f"Formatted: {format_articles_as_text(result)}")

---
## Step 8: Different Symptom Queries

Let's test with different medical scenarios to see the relevance of results.

In [ ]:
test_cases = [
    "severe chest pain, difficulty breathing",
    "nausea vomiting abdominal pain",
    "persistent cough, night sweats, weight loss",
]

for symptoms in test_cases:
    print(f"{'=' * 60}")
    print(f"🔍 Symptoms: {symptoms}")
    print(f"📝 Query: {_build_pubmed_query(symptoms)}")
    print(f"{'=' * 60}")
    
    arts = fetch_pubmed_articles_with_metadata(symptoms)
    print(f"📊 Found {len(arts)} articles:")
    for a in arts:
        print(f"   • {a['title']}")
    print()

---
## Step 9: Full Pipeline — Symptoms → Articles → Summary

This is how it all comes together in the actual Clinisight AI pipeline.

In [ ]:
from functions.symptom_extractor import extract_symptoms
from functions.summerize_pubmed import summarize_text

# Patient input
patient_text = "I have a severe headache and fever, but no nausea. I also feel dizzy."

print("👤 Patient says:", patient_text)
print()

# Step 1: Extract symptoms (LLM-powered)
symptoms = extract_symptoms(patient_text)
print(f"🔬 Extracted symptoms: {symptoms}")
print()

# Step 2: Fetch PubMed evidence
articles = fetch_pubmed_articles_with_metadata(" ".join(symptoms))
print(f"📚 Found {len(articles)} PubMed articles")
for a in articles:
    print(f"   • {a['title']}")
print()

# Step 3: Format articles as text
articles_text = format_articles_as_text(articles)
print(f"📝 Formatted text length: {len(articles_text)} chars")
print()

# Step 4: Summarize with LLM
print("🤖 Generating research summary...")
summary = summarize_text(articles_text[:3000])
print()
print("=" * 60)
print("📋 RESEARCH SUMMARY")
print("=" * 60)
print(summary)

---
## Summary

| Step | Function | What it does |
|------|----------|-------------|
| 1 | `_build_pubmed_query()` | Converts raw symptoms → boolean PubMed query |
| 2 | PubMed `esearch` API | Returns matching article PMIDs |
| 3 | PubMed `efetch` API | Returns full XML with metadata |
| 4 | `_parse_article()` | Extracts title, abstract, authors, date from XML |
| 5 | `format_articles_as_text()` | Converts dicts → clean text for LLM |
| 6 | `summarize_text()` | LLM summarizes the research evidence |

**Key features:**
- 🔄 Retry with exponential backoff (3 attempts)
- 🔑 NCBI API key support (10 req/sec vs 3 req/sec)
- 🛡️ Input validation — empty/garbage queries handled gracefully
- 📝 Structured logging instead of print() in production
- ❌ No fake mock data — honest empty results when nothing found